# LA Airbnb cleaning checks

This notebook checks the cleaning output against the original listings data.

In [9]:
import pandas as pd

raw = pd.read_csv("LA_listings.csv")
cleaned = pd.read_csv("LA_listings_cleaned_yaxin.csv")

print("Original:", raw.shape)
print("Cleaned:", cleaned.shape)

Original: (43932, 19)
Cleaned: (43932, 21)


In [10]:
ids = [2708, 2732, 6033, 9140, 584724]

before = raw.loc[raw["id"].isin(ids), ["id", "price"]].copy()
before["expected_has_price"] = before["price"].notna()
before["expected_high_price_review"] = before["price"] > 1000

after = cleaned.loc[
    cleaned["id"].isin(ids),
    ["id", "has_price", "high_price_review"]
]

check = before.merge(after, on="id")
check["has_price_matches"] = check["expected_has_price"] == check["has_price"]
check["high_price_matches"] = (
    check["expected_high_price_review"] == check["high_price_review"]
)
check

,id,price,expected_has_price,expected_high_price_review,has_price,high_price_review,has_price_matches,high_price_matches
0,2708,67.0,True,False,True,False,True,True
1,2732,214.0,True,False,True,False,True,True
2,6033,100.0,True,False,True,False,True,True
3,9140,NaN,False,False,False,False,True,True
4,584724,1043.0,True,True,True,True,True,True


## Cleaning decisions and checks

| Decision | Reason | Check |
|---|---|---|
| Keep rows with missing price; add `has_price` | A missing price should not be invented, and the listing may still be useful for other analyses. | 5,570 prices are missing; ID 9140 has `has_price = False`. |
| Keep prices above $1,000; add `high_price_review` | An unusually high price needs review, but it is not enough evidence to delete a listing. | 2,493 listings are flagged; ID 584724 ($1,043) is `True`. |
| Keep original listing IDs | IDs are needed to match original and processed records. | 0 duplicate IDs; all five checked IDs appear in the output. |
| Keep `availability_365` values | No values failed the 0–365 range check. | 0 invalid values. |

The five record checks above compare expected flags from the original price with the actual flags in the processed file. Both comparisons match for all five records.